In [1]:
# Cell 1: dựng lại bộ công cụ cho notebook mới
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # kết nối Python với BigQuery

def mb(n):
    # đổi byte sang MB; n là None (job chưa xong) thì trả về None, không ép thành 0
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng, không chạy thật
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)  # chỉ tạo cấu hình
    job = client.query(sql, job_config=cfg)                              # ước lượng xảy ra ở dòng này
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật, tắt cache, giới hạn bytes billed, in processed và billed
    cfg = bigquery.QueryJobConfig(use_query_cache=False,
                                  maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()   # job.result() chờ job xong rồi mới đọc số byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

print("Đã sẵn sàng")

Đã sẵn sàng


In [7]:
# Cell 2: hạn mặc định của từng dataset
sql_ds = f"""
SELECT schema_name, option_name, option_value
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.SCHEMATA_OPTIONS`
WHERE option_name LIKE 'default%expiration%'   -- chỉ lấy các tùy chọn hạn mặc định
ORDER BY schema_name, option_name
"""
check(sql_ds)               # dry run trước
df_ds = stats(sql_ds)       # chạy thật
print(df_ds)

Sẽ quét: 10.0 MB
processed: 10.0 MB | billed: 10.0 MB
Empty DataFrame
Columns: [schema_name, option_name, option_value]
Index: []


In [8]:
# Cell 3: hạn của từng bảng/view, lặp qua 3 dataset
for ds_id in ("lab", "dwh", "raw"):          # mỗi vòng gán một tên dataset vào ds_id
    sql_tb = f"""
    SELECT table_name, option_name, option_value
    FROM `{PROJECT}.{ds_id}.INFORMATION_SCHEMA.TABLE_OPTIONS`
    WHERE option_name IN ('expiration_timestamp', 'partition_expiration_days')
    ORDER BY table_name
    """
    print("Dataset:", ds_id)
    print(stats(sql_tb))                     # stats in processed/billed rồi trả bảng kết quả

Dataset: lab
processed: 10.0 MB | billed: 10.0 MB
Empty DataFrame
Columns: [table_name, option_name, option_value]
Index: []
Dataset: dwh
processed: 10.0 MB | billed: 10.0 MB
Empty DataFrame
Columns: [table_name, option_name, option_value]
Index: []
Dataset: raw
processed: 10.0 MB | billed: 10.0 MB
Empty DataFrame
Columns: [table_name, option_name, option_value]
Index: []


In [4]:
# Cell 4: gỡ hạn tự xóa ở 3 tầng (dataset, bảng/view, partition) cho lab, dwh, raw
for ds_id in ("lab", "dwh", "raw"):                       # mỗi vòng gán một tên dataset vào ds_id
    ds = client.get_dataset(f"{PROJECT}.{ds_id}")         # lấy thông tin dataset
    ds.default_table_expiration_ms = None                 # bỏ hạn mặc định cho bảng mới
    ds.default_partition_expiration_ms = None             # bỏ hạn mặc định cho partition mới
    client.update_dataset(ds, ["default_table_expiration_ms",
                               "default_partition_expiration_ms"])   # chỉ cập nhật hai trường này
    print("Dataset", ds_id, "| đã bỏ hạn mặc định")

    for item in client.list_tables(ds):                   # mỗi vòng gán một bảng hoặc view vào item
        t = client.get_table(item.reference)              # lấy thông tin đầy đủ
        changed = []                                      # danh sách trường sẽ cập nhật
        if t.expires is not None:                         # bảng đang có hạn
            t.expires = None
            changed.append("expires")
        if t.time_partitioning is not None and t.time_partitioning.expiration_ms is not None:
            t.time_partitioning.expiration_ms = None      # partition đang có hạn
            changed.append("time_partitioning")
        try:
            if changed:
                client.update_table(t, changed)           # gửi thay đổi lên BigQuery
            print("  ", t.table_id, "| đã gỡ:", changed)
        except Exception as e:                            # một bảng lỗi thì in ra và đi tiếp
            print("  ", t.table_id, "| LỖI:", str(e)[:150])

Dataset lab | đã bỏ hạn mặc định
   fact_items_clu | đã gỡ: ['expires']
   fact_items_part | đã gỡ: ['time_partitioning']
   fact_items_plain | đã gỡ: ['expires']
   fact_items_recent | đã gỡ: ['expires']
   mv_net_sales_daily | đã gỡ: ['expires']
   v_net_sales | đã gỡ: ['expires']
Dataset dwh | đã bỏ hạn mặc định
   dim_date | đã gỡ: ['expires']
   dim_products | đã gỡ: ['expires']
   dim_users | đã gỡ: ['expires']
   fact_order_items | đã gỡ: ['expires']
Dataset raw | đã bỏ hạn mặc định
   us_states | đã gỡ: ['expires']
   weather_csv_py | đã gỡ: ['expires']
   weather_csv_ui | đã gỡ: ['expires']
   weather_daily | đã gỡ: ['expires']
   weather_parquet_py | đã gỡ: ['expires']
   weather_parquet_ui | đã gỡ: ['expires']


In [5]:
# Cell 5: tạo lại lab.fact_items_part, partition theo tháng, đủ dữ liệu
ddl_part = f"""
CREATE OR REPLACE TABLE `{PROJECT}.lab.fact_items_part`
PARTITION BY DATE_TRUNC(order_date, MONTH)
OPTIONS (require_partition_filter = TRUE)
AS
SELECT order_item_id, order_id, user_id, product_id, order_date_key,
       status, sale_price, cost, gross_profit, order_date    -- liệt kê cột, không dùng SELECT *
FROM `{PROJECT}.lab.fact_items_plain`
"""

client.query(ddl_part).result()      # gửi lệnh rồi chờ xong
print("Đã tạo lại bảng partition")

Đã tạo lại bảng partition


In [6]:
# Cell 6: kiểm tra số dòng (phải có điều kiện trên cột partition vì đã bật require_partition_filter)
sql_cnt = f"""
SELECT COUNT(*) AS so_dong, MIN(order_date) AS ngay_dau, MAX(order_date) AS ngay_cuoi
FROM `{PROJECT}.lab.fact_items_part`
WHERE order_date >= '2019-01-01'
"""
check(sql_cnt)
print(stats(sql_cnt))

Sẽ quét: 1.38 MB
processed: 1.38 MB | billed: 10.0 MB
   so_dong    ngay_dau   ngay_cuoi
0   180771  2019-01-11  2026-10-04
